In [1]:
pip install selenium beautifulsoup4 pandas webdriver-manager

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time
import re
import json

In [2]:
author_ids = [
    "5980049",
    "6904770",
    "6990421",
    "6904479",
    "6921217",
    "6111815",
    "6904403",
    "6727972"]

In [3]:
driver = webdriver.Chrome()

In [ ]:
def scrape_articles(author_id, source):
    url = f"https://sinta.kemdiktisaintek.go.id/authors/profile/{author_id}/?view={source}"
    driver.get(url)
    time.sleep(1)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    publikasi = []
    for item in soup.select("div.ar-list-item.mb-5"):
        judul_tag = item.select_one(".ar-title a")
        judul = judul_tag.get_text(strip=True) if judul_tag else ""
        pub_tag = item.select_one(".ar-pub")
        sumber = pub_tag.get_text(" ", strip=True) if pub_tag else ""
        quartile_tag = item.select_one(".ar-quartile")
        quartile = quartile_tag.get_text(" ", strip=True) if quartile_tag else ""
        tahun = None
        year_tag = item.select_one(".ar-year")
        if year_tag:
            match = re.search(r"\d{4}", year_tag.get_text())
            if match:
                tahun = int(match.group())
        sitasi = 0
        cited_tag = item.select_one(".ar-cited")
        if cited_tag:
            match = re.search(r"\d+", cited_tag.get_text())
            if match:
                sitasi = int(match.group())
        publikasi.append({
            "judul": judul,
            "sumber": sumber,
            "quartile": quartile,
            "tahun": tahun,
            "sitasi": sitasi
        })
    return publikasi

In [ ]:
def scrape_researches(author_id):
    url = f"https://sinta.kemdiktisaintek.go.id/authors/profile/{author_id}/?view=researches"
    driver.get(url)
    time.sleep(1)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    researches = []
    for item in soup.select("div.ar-list-item.mb-5"):
        judul_tag = item.select_one(".ar-title a")
        judul = (
            judul_tag.get_text(strip=True)
            if judul_tag else ""
        )
        leader = ""
        for a in item.select(".ar-meta a"):
            txt = a.get_text(" ", strip=True)
            if txt.startswith("Leader"):
                leader = (
                    txt.replace("Leader :", "")
                    .strip()
                )
        pub_tag = item.select_one(".ar-pub")
        skema = (
            pub_tag.get_text(" ", strip=True)
            if pub_tag else ""
        )
        personil = []
        for a in item.select(
            ".ar-meta a[href*='authors/profile']"
        ):
            nama = a.get_text(strip=True)
            if nama:
                personil.append(nama)
        tahun = None
        year_tag = item.select_one(".ar-year")
        if year_tag:
            match = re.search(
                r"\d{4}",
                year_tag.get_text()
            )
            if match:
                tahun = int(match.group())
        dana = ""
        status = ""
        sumber = ""
        quartiles = item.select(".ar-quartile")
        if len(quartiles) > 0:
            dana = quartiles[0].get_text(
                " ",
                strip=True
            )
        if len(quartiles) > 1:
            status = quartiles[1].get_text(
                " ",
                strip=True
            )
        if len(quartiles) > 2:
            sumber = quartiles[2].get_text(
                " ",
                strip=True
            )
        researches.append({
            "judul": judul,
            "leader": leader,
            "skema": skema,
            "personil": personil,
            "tahun": tahun,
            "dana": dana,
            "status": status,
            "sumber": sumber
        })
    return researches

In [ ]:
def scrape_author(author_id):
    url = f"https://sinta.kemdiktisaintek.go.id/authors/profile/{author_id}"
    driver.get(url)
    time.sleep(2)
    soup = BeautifulSoup(
        driver.page_source,
        "html.parser"
    )
    nama_tag = soup.find("h3")
    if not nama_tag:
        return None
    nama_dosen = nama_tag.get_text(strip=True)
    text = soup.get_text("\n")
    lines = [
        x.strip()
        for x in text.split("\n")
        if x.strip()
    ]
    idx = lines.index(nama_dosen)
    institusi = (
        lines[idx + 1]
        if idx + 1 < len(lines)
        else ""
    )
    prodi = (
        lines[idx + 2]
        if idx + 2 < len(lines)
        else ""
    )
    sinta_id = ""
    if idx + 3 < len(lines):
        match = re.search(
            r"\d+",
            lines[idx + 3]
        )
        if match:
            sinta_id = match.group()
    bidang_keahlian = []
    for item in lines[idx + 4:]:
        if re.match(r"^\d", item):
            break
        bidang_keahlian.append(item)
    sources = [
        "scopus",
        "wos",
        "garuda",
        "googlescholar",
        "rama"
    ]
    all_publications = {}
    for source in sources:
        all_publications[source] = (
            scrape_articles(
                author_id,
                source
            )
        )
    all_researches = (
        scrape_researches(
            author_id
        )
    )
    return {
        "sinta_id": sinta_id,
        "nama_dosen": nama_dosen,
        "institusi": institusi,
        "prodi": prodi,
        "bidang_keahlian": bidang_keahlian,
        "daftar_publikasi": all_publications,
        "daftar_penelitian": all_researches
    }

In [ ]:
all_dosen = []
for author_id in author_ids:
    print(f"Scraping {author_id}")
    try:
        data = scrape_author(author_id)
        if data:
            all_dosen.append(data)
    except Exception as e:
        print(
            f"Gagal {author_id}: {e}"
        )

Scraping 5980049
Scraping 6904770
Scraping 6990421
Scraping 6904479
Scraping 6921217
Scraping 6111815
Scraping 6904403
Scraping 6727972


In [ ]:
with open(
    "dosen_sinta.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        all_dosen,
        f,
        indent=4,
        ensure_ascii=False
    )
driver.quit()
print(f"\nTotal dosen berhasil: {len(all_dosen)}")
print("File tersimpan: dosen_sinta.json")


Total dosen berhasil: 8
File tersimpan: sinta_dosen.json


In [ ]:
with open(
    "dosen_sinta.json",
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)
print(
    json.dumps(
        data,
        indent=4,
        ensure_ascii=False
    )
)

[
    {
        "sinta_id": "5980049",
        "nama_dosen": "DEWI WISNU WARDANI",
        "institusi": "Universitas Sebelas Maret",
        "prodi": "S1 - Sains Data",
        "bidang_keahlian": [
            "Semantic Web",
            "Knowledge Management"
        ],
        "daftar_publikasi": {
            "scopus": [
                {
                    "judul": "An Analysis of an ESP32-Based IoT System for Soil Moisture, Nutrient, and pH Control in Black Betel Cultivation",
                    "sumber": "Proceedings Icsintesa 2025 2025 5th International Conference of Science and Information Technology i",
                    "quartile": "no-Q as Conference Proceedin",
                    "tahun": 2025,
                    "sitasi": 0
                },
                {
                    "judul": "Automatic Semantic Annotation of Indonesian Language Phrase Using N-Gram Language Model",
                    "sumber": "International Journal on Advanced Science Engineering and I